In [1]:
!pip install nltk rouge-score

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [2]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download('wordnet')
nltk.download('punkt')

def compute_bleu(reference, candidate):
    """
    Compute BLEU score for BLEU-1, BLEU-2, BLEU-3, and BLEU-4 between reference and candidate.
    Uses a smoothing function for short sentences.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    smoothie = SmoothingFunction().method1  # Smoothing for short sequences
    
    bleu_scores = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        bleu_scores[f"BLEU-{n}"] = sentence_bleu(reference_tokens, candidate_tokens, weights=weights, smoothing_function=smoothie)
    
    return bleu_scores

def compute_rouge(reference, candidate):
    """
    Compute ROUGE-L, ROUGE-1, and ROUGE-2 scores.
    Returns the F1 scores.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return {k: v.fmeasure for k, v in scores.items()}

def compute_meteor(reference, candidate):
    """
    Compute METEOR score.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    return meteor_score(reference_tokens, candidate_tokens)




[nltk_data] Downloading package wordnet to /Users/wt/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/wt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
import pandas as pd
import os

results_folder = "results"
result_filename = "results/ablation/pororo_ablation_language_gpt_4o_mini.csv"
evaluation_results_folder = "results/evaluation"

os.makedirs(evaluation_results_folder, exist_ok=True)

result = pd.read_csv(result_filename)

os.makedirs(evaluation_results_folder, exist_ok=True)

for i, row in result.iterrows():

    if i == len(result)-1: # avoid the last row with total and average values
        continue

    reference_answer = row["correct_answer"].lower()
    generated_answer = row["predicted_answer"].lower()

    bleus = compute_bleu(reference_answer, generated_answer)
    rouges = compute_rouge(reference_answer, generated_answer)
    meteor =  compute_meteor(reference_answer, generated_answer)
    
    for n in range(1, 5):
        result.at[i, f"BLEU-{n}"] = bleus[f"BLEU-{n}"]
    
    for rouge_type, score in rouges.items():
        result.at[i, f"{rouge_type.upper()}"] = score
    
    result.at[i, "METEOR"] = meteor
    

output_filename = os.path.basename(result_filename)

result.to_csv(os.path.join(evaluation_results_folder, output_filename), index=False)


In [4]:
result

,row_num,video_name,gif_num,qid,question,correct_answer,predicted_answer,evaluator_scores,accuracy,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE1,ROUGE2,ROUGEL,METEOR
0,1,Pororo_ENGLISH1_1_ep1,14,383.0,whtat does eddy ask pororo,he asks pororo what are you doing,eddy asks crong why pororo is acting urgently,"0.25,0.25,0.25",0.25000,0.250000,0.059761,0.039045,0.033032,0.266667,0.000000,0.266667,0.211268
1,2,Pororo_ENGLISH1_1_ep10,12,1100.0,were eddy's friends interested seeing his new ...,"yes, they ran happily towards the new toy.","yes, eddy's friends were interested in seeing ...","1.0,1.0,1.0",1.00000,0.150000,0.028098,0.016369,0.012674,0.275862,0.074074,0.206897,0.163043
2,3,Pororo_ENGLISH1_1_ep10,4,1090.0,what did eddy say after getting the book?,"eddy told, "" what should i make today""",eddy expressed excitement about making a new t...,"0.5,0.5,0.5",0.50000,0.083333,0.027524,0.019640,0.017033,0.210526,0.000000,0.210526,0.119048
3,4,Pororo_ENGLISH1_1_ep11,51,1181.0,why did pororo look to ground?,because he was sorry.,pororo looked to the ground likely due to feel...,"1.0,1.0,1.0",1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,5,Pororo_ENGLISH1_1_ep12,29,1215.0,what exploded in pororo's face,a bomb box exploded pororo's face,a bomb box that crong hid exploded in pororo's...,"1.0,1.0,1.0",1.00000,0.357143,0.234404,0.166054,0.080323,0.636364,0.400000,0.636364,0.655882
5,6,Pororo_ENGLISH1_1_ep12,36,1222.0,what does poby ask when he sees eddy,poby asks eddy why is he so jumpy,"poby asks eddy, ""what are you arguing about","0.25,0.25,0.25",0.25000,0.250000,0.188982,0.084120,0.058739,0.375000,0.285714,0.375000,0.234375
6,7,Pororo_ENGLISH1_1_ep12,43,1226.0,what does confess in loopy's house,eddy confesses he placed the box in pororo's h...,eddy confesses to placing the box that caused ...,"0.75,0.75,0.75",0.75000,0.235294,0.171499,0.058096,0.034401,0.370370,0.240000,0.370370,0.379592
7,8,Pororo_ENGLISH1_1_ep12,49,1232.0,what does pororo say to crong after he realize...,pororo apologizes to crong and says he made a ...,"pororo tells crong, ""you are such a troublemak...","0.5,0.5,0.5",0.50000,0.083333,0.019035,0.011809,0.009410,0.171429,0.000000,0.171429,0.131579
8,9,Pororo_ENGLISH1_1_ep13,12,1258.0,did eddy stay longer after agreeing to sing,"no, he left right away",eddy did not stay longer after agreeing to sin...,"1.0,1.0,1.0",1.00000,0.055556,0.018078,0.012688,0.010802,0.086957,0.000000,0.086957,0.079365
9,10,Pororo_ENGLISH1_1_ep13,41,1283.0,did eddy's entrance impress the audience,"yes, they were all surprised and clapped","eddy's entrance did impress the audience, as i...","1.0,1.0,1.0",1.00000,0.071429,0.023440,0.016605,0.014284,0.090909,0.000000,0.090909,0.064935
